In [5]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import glob
import os
import cmath 
import json
import seaborn as sns
from collections import Counter
import scipy as sp
from scipy.stats import linregress

import PyQt5
# %matplotlib qt
import mne
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from tqdm import tqdm

# **Variables**

In [ ]:
device_list = ["macbook", "iMac"]
device = "iMac"

subject = "09"

SW = 2 # smoothing window size (in seconds)

if device == "iMac" :
    derivative_root = f'/Volumes/My Passport/Data/derivatives/'
elif device == "macbook" :
    erivative_root = f'/Volumes/My Passport/Data/derivatives/'

fname = ?
detrend_order = ?
pen_param = 500
kfold = 10
SUBJS = [f'{i:02d}' for i in range(1, 11)]
tmin = -0.2
tmax = 1.5 

# **Functions**

In [ ]:
## FUNCTION 

def get_timearray(SW, tmin, tmax):
    """when smoothing window applied, the time array might be shifted to the right.
    """
    
    if device == "iMac" :
        epochs = mne.read_epochs(f'/Volumes/My Passport/Data/derivatives/mne-bids-pipeline/sub-{subject}/meg/sub-{subject}_task-FRINT_proc-clean_epo.fif', preload=True)
    elif device == "macbook" :
        epochs = mne.read_epochs(f'/Volumes/My Passport/Data/derivatives/mne-bids-pipeline/sub-{subject}/meg/sub-{subject}_task-FRINT_proc-clean_epo.fif', preload=True)

    times = epochs.times
    if SW > 1:
        times = times - (times[-1] - times[-2]) * SW / 2
    idx_zero = np.abs(times - 0).argmin()
    idx_tmin = np.abs(times - tmin).argmin()
    idx_tmax = np.abs(times - tmax).argmin()
    times = times[int(SW) - 1:]
    return(times)


def fisher_z_transform(r):
    """ apply fisher transformation to avoid non normal data. Since the correlation values are between -1 and 1,
    the fisher transformation is applied to make the data more normal and to allow for better statistical analysis
    
    r is the pearson correlation value
    """
    r = np.array(r)
    return 0.5 * np.log((1 + r) / (1 - r))

def moving_average(x, w):
    return np.convolve(x, np.ones(w)/w, 'valid')



def read_sub(savename, SW):
    """ read the results of the decoding and apply a moving average to smooth the results. 
    The moving average is applied to the predicted values of the decoding, and the true values are not smoothed. 
    The function returns the true values, the smoothed predicted values, and the test ids if they are present in the data.
    """
    
    dat = pd.read_pickle(savename)
    y_pred = np.stack(dat['y_pred'].to_numpy()) 
    y_true = dat['y_test'].to_numpy()[0]
    y_pred_avg = np.array([moving_average(y_pred[:, i], SW) for i in range(len(y_true))])
    if 'test_id' in dat.columns:
        test_id =  np.stack(dat['test_id'].to_numpy())
        return(y_true, y_pred_avg, test_id[0])
    else:
        return(y_true, y_pred_avg)
    
    
def open_decodingresults(savename, SW):
    """ 
    """
    output=[]
    alldat = []
    y_true, y_pred_avg, *_ = read_sub(savename, SW)
    alldat.append([sp.stats.pearsonr(y_true, y_pred_avg[:,i])[0] for i in range(np.shape(y_pred_avg)[1])])
    output.append(fisher_z_transform(alldat))
    return(output)






def Temporal_RidgeDecoding(Data,Y,indexes,pen_param,kfold):
    '''
    :param Data: MEG data to decode from
    :param Y: Value to predict
    :param indexes: index position of the trials
    :param pen_param: penalization parameter for the ridge regression
    :param kfold: number of folds

    Function to compute the Temporal decoding with ridge regression of variable 'Y' from MEG data 'Data'
    For reproducibility the seed is set to 42.
    '''

    corr = [];
    y_predsave = [];
    y_testsave = [];
    res = pd.DataFrame()  # Declare variable
    kf = KFold(n_splits=kfold, shuffle=True, random_state=42)
    test_id = [];
    for train_index, test_index in kf.split(Data[:, :, 0]):
        test_id = [*test_id, *indexes[test_index]]
    for i in tqdm(range(np.shape(Data)[2])):
        X = Data[:, :, i]
        # kf = KFold(n_splits=kfold, shuffle=True, random_state=42)
        corr_fold = []
        y_pred_fold = []
        y_test_fold = []
        for train_index, test_index in kf.split(X):
            X_train, X_test = X[train_index], X[test_index]  # Train data for this fold
            y_train, y_test = Y[train_index], Y[test_index]  # Test data for this fold
            reg = make_pipeline(StandardScaler(with_mean=False), Ridge(alpha=pen_param,))
            reg.fit(X_train, y_train)     # Train
            y_pred = reg.predict(X_test)  # Test
            corr_fold.append(sp.stats.pearsonr(y_test, y_pred)[0])  # Append pearson corr results
            y_pred_fold  = [*y_pred_fold, *y_pred]
            y_test_fold = [*y_test_fold, *y_test]
        y_predsave.append(np.array(y_pred_fold).flatten())
        y_testsave.append(np.array(y_test_fold).flatten())
        corr.append(np.mean(corr_fold))
    res['corr'] = corr
    res['y_pred'] = y_predsave
    res['y_test'] = y_testsave
    res['test_id'] = [test_id for i in range(np.shape(Data)[2])]
    return(res)


# **Prepare Data**

In [ ]:
#### PREPARE DATA (X) AND VARIABLE TO PREDICT (Y)
epochs = mne.read_epochs(derivative_root+fname,preload=True,verbose=False,)
epochs.apply_function(
    mne.filter.detrend,
    picks="meg",
    order=detrend_order
)
epochs = epochs.copy().apply_hilbert(
    envelope=False
)
Data = epochs.get_data()  # Get data
real_part = np.real(Data)
imag_part = np.imag(Data)
Data = np.concatenate((real_part, imag_part), axis=1)

Angle  = ((np.array(epochs.metadata.PresAngle)%180)/180)*np.pi
Y_cos = np.abs((Angle%np.pi) - np.pi / 2)
Y_sin = np.abs(((Angle+np.pi/4) % np.pi)-np.pi/2)

indexes = epochs.metadata.index



# **Call decoding and save**

In [ ]:


####### CALL DECODING AND SAVE
Temporal_RidgeDecoding(Data,Y_cos,indexes,pen_param,kfold).to_pickle(derivative_root + 'sub-' + subject + '/meg/' + 'Decode_PresentedAngles/DecodePresAngle_Cos' + '.pkl')



# **Open and plot decoding results**

In [ ]:



###### OPEN AND PLOT
times = get_timearray(SW, tmin, tmax)
idx_zero = np.abs(times - 0).argmin()
idx_tmin = np.abs(times - tmin).argmin()
idx_tmax = np.abs(times - tmax).argmin()
alldat =[]
for i, sub in enumerate(SUBJS):
    plt.subplot(int(len(SUBJS)/2)+1, 2, i+1)
    plt.plot([times[0],times[-1]],[0,0], 'k:')
    plt.plot([0,0],[-0.2,0.4], 'k:')
    savename = derivative_root + 'sub-' + sub + '/meg/' + 'Decode_RotatedAngles/DecodeRotAngle_Cos' + '.pkl'
    datCos = open_decodingresults(savename, SW)
    savename = derivative_root + 'sub-' + sub + '/meg/' + 'Decode_RotatedAngles/DecodeRotAngle_Sin' + '.pkl'
    datSin = open_decodingresults(savename, SW)
    dat = np.mean([datCos,datSin],0)
    alldat.append(np.array(dat).flatten())
    plt.plot(times, np.array(datCos)[0,0,:], 'red', linewidth=1)
    plt.plot(times, np.array(datSin)[0,0,:], 'green', linewidth=1)
    plt.plot(times, np.array(dat)[0,0,:], 'black', linewidth=2)
    plt.ylim([-0.1,0.2])
    plt.title(sub)
plt.tight_layout()
plt.show()



